# Classical Cryptography — Five Cipher Implementations

This notebook covers five classical cipher techniques with full encryption and decryption:

| # | Cipher | Type |
|---|--------|------|
| 1 | Caesar | Substitution (monoalphabetic) |
| 2 | Vigenère | Substitution (polyalphabetic) |
| 3 | Playfair | Substitution (digraph) |
| 4 | Hill (2×2 manual) | Polygraphic substitution |
| 4b | Hill (n×n NumPy) | Polygraphic substitution |
| 5 | Rail Fence | Transposition |

Each cipher is self-contained — no shared helpers. Every non-obvious line is explained.

In [ ]:
# Shared test inputs used across all ciphers
PLAINTEXT  = "HELLO WORLD"
KEYWORD    = "YASH"

---
## 1. Caesar Cipher

**Idea:** Shift every letter in the alphabet by a fixed number of positions.

```
Encrypt: E(x) = (x + shift) mod 26
Decrypt: D(x) = (x - shift) mod 26
```

- Only letters are shifted; spaces and punctuation pass through unchanged.
- Decryption is identical to encryption with a negative shift.
- Classic shift is 3 (ROT-3); ROT-13 is also common.

**Example:**  
`H (7) + 3 = K (10)` → K  
`K (10) - 3 = H (7)` → H

In [ ]:
def caesar_encrypt(plaintext, shift):
    """
    Shift every alphabetic character forward by `shift` positions.
    Non-alphabetic characters (spaces, punctuation) are kept as-is.
    """
    ciphertext = ""

    for character in plaintext:
        if character.isalpha():
            # Uppercase and lowercase letters have different ASCII base values.
            # 'A' = 65, 'a' = 97.  We subtract the base to get a 0-25 index,
            # apply the shift modulo 26 to wrap around, then add the base back.
            ascii_base = ord('A') if character.isupper() else ord('a')
            letter_index = ord(character) - ascii_base          # 0..25
            shifted_index = (letter_index + shift) % 26         # wrap at Z
            ciphertext += chr(shifted_index + ascii_base)
        else:
            ciphertext += character   # preserve spaces, digits, punctuation

    return ciphertext


def caesar_decrypt(ciphertext, shift):
    """
    Reverse the Caesar shift by subtracting instead of adding.
    Python's modulo handles negative results correctly (e.g. -3 % 26 = 23),
    so this is safe without extra conditionals.
    """
    plaintext = ""

    for character in ciphertext:
        if character.isalpha():
            ascii_base = ord('A') if character.isupper() else ord('a')
            letter_index  = ord(character) - ascii_base
            original_index = (letter_index - shift) % 26    # subtract to reverse
            plaintext += chr(original_index + ascii_base)
        else:
            plaintext += character

    return plaintext


# ── Test ─────────────────────────────────────────────────────────────────────
SHIFT = 3
caesar_enc = caesar_encrypt(PLAINTEXT, SHIFT)
caesar_dec = caesar_decrypt(caesar_enc, SHIFT)

print(f"Plaintext  : {PLAINTEXT}")
print(f"Encrypted  : {caesar_enc}")
print(f"Decrypted  : {caesar_dec}")
print(f"Round-trip : {'✓ PASS' if caesar_dec == PLAINTEXT else '✗ FAIL'}")

---
## 2. Vigenère Cipher

**Idea:** Use a repeating keyword to assign a different Caesar shift to each letter of the plaintext. This breaks the fixed-shift weakness of Caesar.

```
Encrypt: E(x_i) = (x_i + k_{i mod len(key)}) mod 26
Decrypt: D(y_i) = (y_i - k_{i mod len(key)}) mod 26
```

where `k_j` is the alphabetic position (0-25) of the j-th key letter.

**Example with key `YASH`:**
```
H + Y(24) = F
E + A(0)  = E
L + S(18) = D
L + H(7)  = S
O + Y(24) = M   ← key wraps back to Y
...
```

In [ ]:
def vigenere_encrypt(plaintext, keyword):
    """
    Each alphabetic character in the plaintext is Caesar-shifted by the
    corresponding letter in the keyword (cycling).  Non-alpha characters
    do NOT advance the keyword index — the key only steps on letters.
    """
    # Build the list of numeric shifts from the keyword (0-25 per letter).
    # We upper-case and filter non-alpha just in case the keyword has spaces.
    key_shifts = [ord(letter.upper()) - ord('A') for letter in keyword if letter.isalpha()]
    key_length = len(key_shifts)

    ciphertext = ""
    key_index  = 0   # tracks our position in the cycling keyword

    for character in plaintext:
        if character.isalpha():
            ascii_base     = ord('A') if character.isupper() else ord('a')
            letter_index   = ord(character) - ascii_base
            shift          = key_shifts[key_index % key_length]  # cycle keyword
            shifted_index  = (letter_index + shift) % 26
            ciphertext    += chr(shifted_index + ascii_base)
            key_index     += 1   # only advance key position for actual letters
        else:
            ciphertext += character

    return ciphertext


def vigenere_decrypt(ciphertext, keyword):
    """
    Subtract the keyword shifts instead of adding them.
    Everything else mirrors the encrypt function exactly.
    """
    key_shifts = [ord(letter.upper()) - ord('A') for letter in keyword if letter.isalpha()]
    key_length = len(key_shifts)

    plaintext = ""
    key_index = 0

    for character in ciphertext:
        if character.isalpha():
            ascii_base      = ord('A') if character.isupper() else ord('a')
            letter_index    = ord(character) - ascii_base
            shift           = key_shifts[key_index % key_length]
            original_index  = (letter_index - shift) % 26   # subtract to undo
            plaintext      += chr(original_index + ascii_base)
            key_index      += 1
        else:
            plaintext += character

    return plaintext


# ── Test ─────────────────────────────────────────────────────────────────────
vigenere_enc = vigenere_encrypt(PLAINTEXT, KEYWORD)
vigenere_dec = vigenere_decrypt(vigenere_enc, KEYWORD)

print(f"Plaintext  : {PLAINTEXT}")
print(f"Keyword    : {KEYWORD}")
print(f"Encrypted  : {vigenere_enc}")
print(f"Decrypted  : {vigenere_dec}")
print(f"Round-trip : {'✓ PASS' if vigenere_dec == PLAINTEXT else '✗ FAIL'}")

---
## 3. Playfair Cipher (Wheatstone–Playfair)

**Idea:** Encrypt pairs of letters (digraphs) using a 5×5 grid built from a keyword.  
J is merged into I (standard convention — only 25 slots available).

### Grid Construction
1. Write the keyword letters in order (no duplicates).
2. Append the remaining alphabet (minus J).
3. Fill a 5×5 matrix left-to-right, top-to-bottom.

### Digraph Preparation Rules
- Ignore non-alpha; uppercase all; J → I.
- Split into pairs. If both letters of a pair are the same, insert **X** between them.
- If the total length is odd, append **X**.

### Encryption Rules (per digraph)
| Case | Rule |
|------|------|
| Same row | Each letter → letter to its right (wrap) |
| Same column | Each letter → letter below it (wrap) |
| Rectangle | Each letter → same row, partner's column |

Decryption reverses directions (left / up instead of right / down).

In [ ]:
def build_playfair_grid(keyword):
    """
    Build the 5x5 Playfair key square from the keyword.

    Returns:
        grid_string : 25-character string, row-major order
        char_to_pos : dict mapping each letter → (row, col)
        pos_to_char : dict mapping (row, col) → letter
    """
    # dict.fromkeys preserves insertion order and drops duplicates (Python 3.7+)
    # We replace J with I everywhere — the grid has no J slot.
    alphabet_without_j = "ABCDEFGHIKLMNOPQRSTUVWXYZ"   # 25 letters
    keyword_cleaned    = keyword.upper().replace("J", "I")
    full_sequence      = keyword_cleaned + alphabet_without_j
    # dict.fromkeys on the combined string drops any letter seen a second time
    grid_string        = "".join(dict.fromkeys(full_sequence))

    char_to_pos = {}   # letter  → (row, col)
    pos_to_char = {}   # (row, col) → letter

    for index, letter in enumerate(grid_string):
        row = index // 5
        col = index %  5
        char_to_pos[letter]      = (row, col)
        pos_to_char[(row, col)]  = letter

    return grid_string, char_to_pos, pos_to_char


def prepare_playfair_digraphs(plaintext):
    """
    Clean the plaintext and split it into valid Playfair digraph pairs.

    Rules applied:
    1. Keep only alpha chars, uppercase, replace J → I.
    2. Walk through and pair letters:
       - If the two letters in a potential pair are equal, insert 'X' as filler
         and push the second letter to the next pair.
    3. If one letter is left unpaired at the end, append 'X'.
    """
    cleaned = [ch.upper().replace("J", "I") for ch in plaintext
               if ch.isalpha()]
    # 'replace' on a single char returns the replacement string;
    # since J→I is one char, this is safe inside a list comprehension.

    digraphs = []
    position = 0

    while position < len(cleaned):
        first_letter = cleaned[position]

        if position + 1 < len(cleaned):
            second_letter = cleaned[position + 1]
        else:
            second_letter = "X"   # pad odd-length text

        if first_letter == second_letter:
            # Same-letter pair: use 'X' as the filler second letter,
            # and DO NOT consume the second original letter yet.
            digraphs.append((first_letter, "X"))
            position += 1   # advance by 1 only
        else:
            digraphs.append((first_letter, second_letter))
            position += 2   # consume both letters

    return digraphs


def playfair_encrypt(plaintext, keyword):
    """Encrypt using the Playfair digraph substitution rules."""
    _, char_to_pos, pos_to_char = build_playfair_grid(keyword)
    digraphs  = prepare_playfair_digraphs(plaintext)
    ciphertext = ""

    for (letter_a, letter_b) in digraphs:
        row_a, col_a = char_to_pos[letter_a]
        row_b, col_b = char_to_pos[letter_b]

        if row_a == row_b:
            # Same row → shift columns RIGHT by 1, wrap with % 5
            ciphertext += pos_to_char[(row_a, (col_a + 1) % 5)]
            ciphertext += pos_to_char[(row_b, (col_b + 1) % 5)]

        elif col_a == col_b:
            # Same column → shift rows DOWN by 1, wrap with % 5
            ciphertext += pos_to_char[((row_a + 1) % 5, col_a)]
            ciphertext += pos_to_char[((row_b + 1) % 5, col_b)]

        else:
            # Rectangle → each letter takes the same row but the OTHER letter's column
            ciphertext += pos_to_char[(row_a, col_b)]
            ciphertext += pos_to_char[(row_b, col_a)]

    return ciphertext


def playfair_decrypt(ciphertext, keyword):
    """
    Decrypt Playfair by reversing the shift directions.
    Same row  → shift LEFT  (-1 instead of +1)
    Same col  → shift UP    (-1 instead of +1)
    Rectangle → unchanged (swapping columns is its own inverse)
    """
    _, char_to_pos, pos_to_char = build_playfair_grid(keyword)
    # Ciphertext is always clean alpha — no need for the digraph preparation logic.
    # We just pair consecutive letters directly.
    digraphs = [(ciphertext[i], ciphertext[i + 1])
                for i in range(0, len(ciphertext), 2)]
    plaintext = ""

    for (letter_a, letter_b) in digraphs:
        row_a, col_a = char_to_pos[letter_a]
        row_b, col_b = char_to_pos[letter_b]

        if row_a == row_b:
            plaintext += pos_to_char[(row_a, (col_a - 1) % 5)]   # LEFT
            plaintext += pos_to_char[(row_b, (col_b - 1) % 5)]

        elif col_a == col_b:
            plaintext += pos_to_char[((row_a - 1) % 5, col_a)]   # UP
            plaintext += pos_to_char[((row_b - 1) % 5, col_b)]

        else:
            # Rectangle case is symmetric — same swap undoes itself
            plaintext += pos_to_char[(row_a, col_b)]
            plaintext += pos_to_char[(row_b, col_a)]

    return plaintext


# ── Test ─────────────────────────────────────────────────────────────────────
playfair_enc = playfair_encrypt(PLAINTEXT, KEYWORD)
playfair_dec = playfair_decrypt(playfair_enc, KEYWORD)

print(f"Plaintext  : {PLAINTEXT}")
print(f"Keyword    : {KEYWORD}")
print(f"Encrypted  : {playfair_enc}")
print(f"Decrypted  : {playfair_dec}")
print()
print("Note: Decrypted output may contain filler 'X' characters")
print("inserted to split repeated-letter pairs — this is expected.")

---
## 4a. Hill Cipher — 2×2 Key, Manual Inverse

**Idea:** Treat pairs of letters as 2D vectors and multiply them by a 2×2 key matrix modulo 26.

```
Encrypt: C = K · P  (mod 26)   where P is a column vector of letter indices
Decrypt: P = K⁻¹ · C  (mod 26)
```

### 2×2 Modular Inverse Formula

For matrix `K = [[a, b], [c, d]]`:
```
det       = a·d - b·c  (mod 26)
det_inv   = modular multiplicative inverse of det  (pow(det, -1, 26))
K⁻¹ (mod 26) = det_inv · [[d, -b], [-c, a]]  (mod 26)
```

The key matrix must have `gcd(det, 26) == 1` (i.e. det must be coprime with 26).

**Key used:** `[[3, 3], [2, 5]]`  
**det = 3·5 − 3·2 = 9**, and gcd(9, 26) = 1 ✓

In [ ]:
def text_to_indices(text):
    """Convert uppercase text to a list of 0-25 indices, ignoring non-alpha."""
    return [ord(ch.upper()) - ord('A') for ch in text if ch.isalpha()]


def indices_to_text(indices):
    """Convert a list of 0-25 indices back to uppercase letters."""
    return "".join(chr(index + ord('A')) for index in indices)


def invert_2x2_key_matrix_mod26(key_matrix):
    """
    Compute the modular inverse of a 2×2 matrix under mod 26.

    Standard 2×2 inverse formula (real numbers):
        inv = (1/det) * [[d, -b], [-c, a]]
    
    Mod 26 version: replace '1/det' with 'pow(det, -1, 26)'
    Python's built-in pow(x, -1, m) computes the modular multiplicative inverse.
    """
    a = key_matrix[0][0]
    b = key_matrix[0][1]
    c = key_matrix[1][0]
    d = key_matrix[1][1]

    determinant = (a * d - b * c) % 26

    # pow(determinant, -1, 26) raises ValueError if gcd(det, 26) != 1
    determinant_inverse = pow(determinant, -1, 26)

    # Build the adjugate (classical adjoint) matrix, then scale by det_inv.
    # Negatives are handled by % 26 (e.g. -3 % 26 = 23 in Python).
    inverse_matrix = [
        [(determinant_inverse *  d) % 26,   (determinant_inverse * -b) % 26],
        [(determinant_inverse * -c) % 26,   (determinant_inverse *  a) % 26],
    ]
    return inverse_matrix


def multiply_2x2_matrix_with_vector(matrix, vector_pair):
    """
    Multiply a 2×2 matrix by a 2-element column vector, result mod 26.
    Returns a list of two encrypted/decrypted letter indices.
    """
    top_row    = matrix[0]
    bottom_row = matrix[1]
    x, y       = vector_pair

    result_top    = (top_row[0]    * x + top_row[1]    * y) % 26
    result_bottom = (bottom_row[0] * x + bottom_row[1] * y) % 26
    return [result_top, result_bottom]


def hill_2x2_encrypt(plaintext, key_matrix):
    """
    Encrypt using a 2×2 Hill cipher key matrix.

    Process:
    1. Convert letters to 0-25 indices.
    2. Pad with 'X' (index 23) if text length is odd.
    3. Process pairs: multiply each 2-element vector by key_matrix mod 26.
    4. Convert resulting indices back to letters.
    """
    letter_indices = text_to_indices(plaintext)

    # Hill cipher needs even-length input; pad with X (index 23) if needed
    if len(letter_indices) % 2 != 0:
        letter_indices.append(23)   # 'X'

    encrypted_indices = []

    for i in range(0, len(letter_indices), 2):
        pair              = letter_indices[i : i + 2]
        encrypted_pair    = multiply_2x2_matrix_with_vector(key_matrix, pair)
        encrypted_indices.extend(encrypted_pair)

    return indices_to_text(encrypted_indices)


def hill_2x2_decrypt(ciphertext, key_matrix):
    """
    Decrypt using the modular inverse of the key matrix.
    Structurally identical to encrypt — only the matrix changes.
    """
    inverse_key_matrix = invert_2x2_key_matrix_mod26(key_matrix)
    letter_indices     = text_to_indices(ciphertext)
    decrypted_indices  = []

    for i in range(0, len(letter_indices), 2):
        pair             = letter_indices[i : i + 2]
        decrypted_pair   = multiply_2x2_matrix_with_vector(inverse_key_matrix, pair)
        decrypted_indices.extend(decrypted_pair)

    return indices_to_text(decrypted_indices)


# ── Test ─────────────────────────────────────────────────────────────────────
KEY_MATRIX_2X2 = [[3, 3],
                  [2, 5]]

hill_enc = hill_2x2_encrypt(PLAINTEXT, KEY_MATRIX_2X2)
hill_dec = hill_2x2_decrypt(hill_enc, KEY_MATRIX_2X2)

print(f"Plaintext  : {PLAINTEXT}")
print(f"Key matrix : {KEY_MATRIX_2X2}")
print(f"Encrypted  : {hill_enc}")
print(f"Decrypted  : {hill_dec}")
print(f"Round-trip : {'✓ PASS' if hill_dec.startswith(''.join(PLAINTEXT.split())) else '✗ FAIL'}")
print()
print("Note: Decrypted output strips spaces and may have trailing padding 'X'.")

---
## 4b. Hill Cipher — n×n Key, NumPy Implementation

**Why NumPy?**  
For key sizes larger than 2×2, computing the modular matrix inverse manually becomes tedious. NumPy lets us:
- Represent the key as an `ndarray`.
- Use `np.linalg.det` for the determinant.
- Use `np.linalg.inv` for the real-valued inverse, then convert to mod-26 integers.

**Modular matrix inverse approach:**
```
K⁻¹ (mod 26) = round(det(K) * inv(K)) * det_inv  (mod 26)
```
This works because `det(K) * inv(K)` gives the adjugate matrix (integer-valued), and dividing out the determinant modularly gives the mod inverse.

The key size `n` determines the block size — the text is split into `n`-letter groups.

In [ ]:
import numpy as np


def compute_modular_matrix_inverse(key_matrix_np, modulus=26):
    """
    Compute the modular inverse of an n×n integer matrix.

    Steps:
    1. Compute the determinant with np.linalg.det (returns a float).
    2. Round and cast to int (floating-point math introduces tiny errors).
    3. Find the modular inverse of the determinant using pow(det, -1, mod).
    4. The adjugate matrix = round(det * np.linalg.inv(K)).
       (np.linalg.inv gives the real-valued inverse; multiplying by det clears
        the denominator, leaving the integer adjugate.)
    5. Scale adjugate by det_inv mod modulus to get K⁻¹ mod modulus.
    """
    key_size = key_matrix_np.shape[0]

    # Step 1-2: integer determinant
    determinant = round(np.linalg.det(key_matrix_np)) % modulus

    # Step 3: modular inverse of the determinant
    determinant_mod_inverse = pow(int(determinant), -1, modulus)

    # Step 4: adjugate = det * K⁻¹ in real space, rounded to ints
    # np.linalg.inv can have floating-point noise, so we round before casting
    real_inverse  = np.linalg.inv(key_matrix_np)
    adjugate      = np.round(np.linalg.det(key_matrix_np) * real_inverse).astype(int)

    # Step 5: scale by the modular det inverse, take mod
    inverse_matrix_mod = (determinant_mod_inverse * adjugate) % modulus
    return inverse_matrix_mod


def hill_nxn_encrypt(plaintext, key_matrix, modulus=26):
    """
    Hill cipher encryption supporting an n×n key matrix.

    The block size equals the matrix dimension n.
    Text is split into n-letter blocks; each block is a column vector
    multiplied by the key matrix.
    """
    key_matrix_np = np.array(key_matrix)
    block_size    = key_matrix_np.shape[0]   # n = number of rows (= cols)

    letter_indices = [ord(ch.upper()) - ord('A') for ch in plaintext if ch.isalpha()]

    # Pad with 'X' (23) to make the total length a multiple of block_size
    remainder = len(letter_indices) % block_size
    if remainder != 0:
        padding_needed = block_size - remainder
        letter_indices.extend([23] * padding_needed)   # 'X' = 23

    encrypted_indices = []

    for block_start in range(0, len(letter_indices), block_size):
        # Extract one block as a column vector (shape: n×1)
        block_vector = np.array(letter_indices[block_start : block_start + block_size])

        # Matrix multiplication: (n×n) @ (n,) → (n,), then mod 26
        # NumPy's @ operator performs matrix/vector multiplication
        encrypted_block = (key_matrix_np @ block_vector) % modulus
        encrypted_indices.extend(encrypted_block.tolist())

    return "".join(chr(int(idx) + ord('A')) for idx in encrypted_indices)


def hill_nxn_decrypt(ciphertext, key_matrix, modulus=26):
    """
    Hill cipher decryption: multiply ciphertext blocks by the modular inverse matrix.
    """
    key_matrix_np    = np.array(key_matrix)
    block_size       = key_matrix_np.shape[0]
    inverse_matrix   = compute_modular_matrix_inverse(key_matrix_np, modulus)

    letter_indices   = [ord(ch.upper()) - ord('A') for ch in ciphertext if ch.isalpha()]
    decrypted_indices = []

    for block_start in range(0, len(letter_indices), block_size):
        block_vector     = np.array(letter_indices[block_start : block_start + block_size])
        decrypted_block  = (inverse_matrix @ block_vector) % modulus
        decrypted_indices.extend(decrypted_block.tolist())

    return "".join(chr(int(idx) + ord('A')) for idx in decrypted_indices)


# ── Test with 2×2 key ────────────────────────────────────────────────────────
KEY_2X2 = [[3, 3],
           [2, 5]]

hill_np_enc_2 = hill_nxn_encrypt(PLAINTEXT, KEY_2X2)
hill_np_dec_2 = hill_nxn_decrypt(hill_np_enc_2, KEY_2X2)
print("=== 2×2 Key ===")
print(f"Plaintext  : {PLAINTEXT}")
print(f"Encrypted  : {hill_np_enc_2}")
print(f"Decrypted  : {hill_np_dec_2}")

# ── Test with 3×3 key ────────────────────────────────────────────────────────
# det([[6,24,1],[13,16,10],[20,17,15]]) = 441; gcd(441,26)=1  ✓
KEY_3X3 = [[ 6, 24,  1],
           [13, 16, 10],
           [20, 17, 15]]

hill_np_enc_3 = hill_nxn_encrypt(PLAINTEXT, KEY_3X3)
hill_np_dec_3 = hill_nxn_decrypt(hill_np_enc_3, KEY_3X3)
print()
print("=== 3×3 Key ===")
print(f"Plaintext  : {PLAINTEXT}")
print(f"Encrypted  : {hill_np_enc_3}")
print(f"Decrypted  : {hill_np_dec_3}")

---
## 5. Rail Fence Transposition Cipher

**Idea:** Write the plaintext in a **zigzag** pattern across `n` rails, then read off each rail top-to-bottom. The letters are rearranged — not substituted — so this is a **transposition** cipher.

```
Rails = 3, Plaintext = HELLOWORLD

Rail 0:  H . . . O . . . L .
Rail 1:  . E . L . W . R . D
Rail 2:  . . L . . . O . . .

Ciphertext = HOLEELWRDLO
```

The zigzag pattern produces a repeating index sequence:
- 3 rails → sequence: `0, 1, 2, 1, 0, 1, 2, 1, ...` (period = 2*(rails-1))

Decryption reconstructs which character goes on which rail, then reads in order.

In [ ]:
def build_zigzag_rail_sequence(num_rails, text_length):
    """
    Generate the rail index for each position in the text.

    The zigzag goes: 0, 1, 2, ..., n-1, n-2, ..., 1, 0, 1, 2, ...
    One full cycle = [0, 1, ..., n-1, n-2, ..., 1]  (length = 2*(n-1))

    We build one period, then tile it across the text using modulo.
    """
    if num_rails == 1:
        # Edge case: single rail — everything goes on rail 0
        return [0] * text_length

    # Build one period of the zigzag pattern
    descending = list(range(num_rails))          # [0, 1, 2, ..., n-1]
    ascending  = list(range(num_rails - 2, 0, -1))  # [n-2, ..., 1]  (no endpoints)
    one_period = descending + ascending          # full up-down cycle
    period_length = len(one_period)              # = 2 * (num_rails - 1)

    # For each text position, look up which rail it belongs to
    rail_sequence = [one_period[position % period_length]
                     for position in range(text_length)]
    return rail_sequence


def rail_fence_encrypt(plaintext, num_rails):
    """
    Distribute plaintext characters across rails following the zigzag pattern,
    then concatenate all rails top-to-bottom to form the ciphertext.
    """
    rail_sequence = build_zigzag_rail_sequence(num_rails, len(plaintext))

    # Initialize each rail as an empty string
    rails = [""] * num_rails

    for character, rail_index in zip(plaintext, rail_sequence):
        rails[rail_index] += character   # place character on its rail

    # Read all rails in order to form the ciphertext
    return "".join(rails)


def rail_fence_decrypt(ciphertext, num_rails):
    """
    Reconstruct the original text from the rail-fence ciphertext.

    Strategy:
    1. Compute the zigzag sequence to know which rail each position belongs to.
    2. Count how many characters land on each rail.
    3. Slice the ciphertext to assign the correct segment to each rail.
    4. For each original position, pull the next character from the matching rail.
    """
    text_length   = len(ciphertext)
    rail_sequence = build_zigzag_rail_sequence(num_rails, text_length)

    # Step 2: count how many characters each rail receives
    characters_per_rail = [0] * num_rails
    for rail_index in rail_sequence:
        characters_per_rail[rail_index] += 1

    # Step 3: slice ciphertext into rail-sized chunks (stored as lists for pop)
    # We use lists so we can efficiently pull characters from the front (pop(0))
    rail_contents = []
    slice_start   = 0
    for count in characters_per_rail:
        rail_contents.append(list(ciphertext[slice_start : slice_start + count]))
        slice_start += count

    # Step 4: rebuild the original text position by position
    # For each position, we know which rail it came from; we take the next
    # character off the front of that rail's list.
    plaintext = ""
    for rail_index in rail_sequence:
        plaintext += rail_contents[rail_index].pop(0)

    return plaintext


# ── Test ─────────────────────────────────────────────────────────────────────
NUM_RAILS = 4

rail_enc = rail_fence_encrypt(PLAINTEXT, NUM_RAILS)
rail_dec = rail_fence_decrypt(rail_enc, NUM_RAILS)

print(f"Plaintext  : {PLAINTEXT}")
print(f"Rails      : {NUM_RAILS}")
print(f"Encrypted  : {rail_enc}")
print(f"Decrypted  : {rail_dec}")
print(f"Round-trip : {'✓ PASS' if rail_dec == PLAINTEXT else '✗ FAIL'}")

# ── Visualise zigzag ─────────────────────────────────────────────────────────
print("\n--- Zigzag Pattern Visualisation ---")
rail_sequence = build_zigzag_rail_sequence(NUM_RAILS, len(PLAINTEXT))
for rail in range(NUM_RAILS):
    row = ""
    for position, character in enumerate(PLAINTEXT):
        row += character if rail_sequence[position] == rail else "."
    print(f"Rail {rail}: {row}")

---
## Summary

| Cipher | Key | Enc input | Operation |
|--------|-----|-----------|----------|
| Caesar | shift (int) | any text | `(x + shift) % 26` per letter |
| Vigenère | keyword (str) | any text | cycling Caesar shifts from keyword |
| Playfair | keyword (str) | alpha only | digraph substitution via 5×5 grid |
| Hill 2×2 | 2×2 matrix | alpha only | matrix multiply pairs mod 26 |
| Hill n×n | n×n matrix | alpha only | matrix multiply n-blocks mod 26 |
| Rail Fence | num_rails (int) | any text | zigzag transposition |

**Key limitations to remember for your exam:**
- Playfair: J is always merged into I; repeated pairs get X inserted.
- Hill: key matrix determinant must be coprime with 26.
- Rail Fence: spaces and punctuation are preserved as-is (in this implementation).